In [ ]:
import math
import numpy as np
import pandas as pd
import scipy.stats as st
import pytensor.tensor as pt
from pytensor.scan import scan, reduce
from pytensor.compile import function

In [ ]:
def LBA(rt, response, n, A, b, v, s, tau=0):
    """PDF of Linear Ballistic Accumulator model."""

    normpdf = lambda x: pt.exp(-0.5 * x**2) / pt.sqrt(2 * math.pi)
    normcdf = lambda x: 0.5 * (1 + pt.erf(x / math.sqrt(2)))

    def tpdf(t, A, b, v, s):
        """First-passage time probability density function for single accumulator."""
        g = (b - A - t * v) / (t * s)
        h = (b - t * v) / (t * s)
        
        # First-passage density formula
        pdf = (1/A) * (-v * normcdf(g) + s * normpdf(g) + v * normcdf(h) - s * normpdf(h))

        return pt.maximum(pdf, 1e-20)  # Numerical stability

    def tcdf(t, A, b, v, s):
        """First-passage time cumulative distribution function for single accumulator."""
        
        # Calculate standardized variables
        g = (b - A - t * v) / (t * s)
        h = (b - t * v) / (t * s)
        
        # CDF components
        p1 = ((b - A - t * v) / A) * normcdf(g)
        p2 = ((b - t * v) / A) * normcdf(h)
        p3 = ((t * s) / A) * normpdf(g)
        p4 = ((t * s) / A) * normpdf(h)
        
        cdf = 1 + p1 - p2 + p3 - p4

        # Ensure CDF is between 0 and 1
        return pt.clip(cdf, 1e-20, 1 - 1e-20)

    # Subtract non-decision time
    t = rt - tau
    
    # 1) First‐passage densities for each accumulator: shape (m, n_trials)
    f = pt.stack([tpdf(t, A, b, vi, s) for vi in v], axis=0)

    # 2) CDFs for each accumulator
    F = pt.stack([tcdf(t, A, b, vi, s) for vi in v], axis=0)

    # 3) Probability that none ever finish
    p_zero = pt.prod(normcdf(-v / s))

    # 4) For each i: f_i * ∏_{j≠i} (1 − F_j)
    numer = pt.stack([
        f[i] * pt.prod(1 - pt.concatenate([F[:i], F[i+1:]]), axis=0)
        for i in range(n)
    ], axis=0)

    # 5) Select the density for the chosen accumulator and normalize
    pdf = numer[response, pt.arange(t.shape[0])] / (1 - p_zero)

    # 6) Zero‐out any non‐positive RTs
    return pt.switch(pt.gt(t, 0), pdf, 1e-20)